# Paired RNA/ATAC preprocessing and split notebook

This notebook is rewritten to follow the **R workflow input format** you provided:

- read from **10x multiome H5**: `pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5`
- keep the same **train / validation split** and **partial modality masking** logic
- generate the same downstream split files

At the same time, it also supports the **four `.h5ad` base files** produced by the earlier Python preprocessing workflow, so you can choose either input mode. This notebook is based on the R pipeline structure you shared earlier. fileciteturn1file0


## Supported input modes

### Mode 1: `from_10x_h5`
Use the same style as your R code:
- `INPUT_H5`
- `FRAGMENTS_FILE` (kept as a parameter for compatibility; not required by the current Python preprocessing logic)
- `GTF_PATH`

This mode will:
1. read 10x multiome H5
2. split RNA and ATAC
3. build the four base files
4. generate split files

### Mode 2: `from_existing_h5ad`
Use already-generated base files:
- `RNA_counts_qc.h5ad`
- `ATAC_counts_qc.h5ad`
- `ATAC_gas.h5ad`
- `feature_aligned.h5ad`

This mode will skip raw preprocessing and directly generate split files.


In [ ]:
# 0. Install if needed
# !pip install scanpy anndata episcanpy scipy pandas numpy scikit-learn h5py jupyter


In [1]:
# 1. Imports

from __future__ import annotations

import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.model_selection import train_test_split


In [8]:
# 2. Parameters
# -------------------------
# Choose input mode:
#   INPUT_MODE = "from_10x_h5"
#   INPUT_MODE = "from_existing_h5ad"
# -------------------------

INPUT_MODE = "from_10x_h5"

# ----- Mode 1: same style as R input -----
INPUT_H5 = "raw_input/pbmc_granulocyte_sorted_10k_filtered_feature_bc_matrix.h5"
FRAGMENTS_FILE = "raw_input/pbmc_granulocyte_sorted_10k_atac_fragments.tsv.gz"   # optional here
GTF_PATH = "raw_input/gencode.v38.primary_assembly.annotation.gtf"

# ----- Mode 2: use existing base files -----
EXISTING_BASE_DIR = "/path/to/existing_base_dir"

# ----- Common output -----
OUTPUT_DIR = "./output_pbmc_multiome"
BATCH_KEY = "batch"
MT_PREFIX = "MT-"

SEED = 1234
VAL_FRAC = 0.2
SINGLE_FRACS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
RNA_KEEP_PROB = 0.5

MIN_TRAIN_RNA_CELLS = 50
MIN_VAL_QUERY_CELLS = 20
MIN_COMMON_FEATURES = 50
SPLIT_DIR_NAME = "results_ratio_loop"


In [9]:
# 3. Utilities

def set_seed(seed: int = 1234) -> None:
    random.seed(seed)
    np.random.seed(seed)

def ensure_dir(path: os.PathLike) -> None:
    Path(path).mkdir(parents=True, exist_ok=True)

def safe_write_h5ad(adata: ad.AnnData, path: os.PathLike) -> None:
    import numpy as np

    if isinstance(adata.X, np.matrix):
        adata.X = np.asarray(adata.X)

    for layer_name in list(adata.layers.keys()):
        if isinstance(adata.layers[layer_name], np.matrix):
            adata.layers[layer_name] = np.asarray(adata.layers[layer_name])

    adata.write(str(path))

def tfidf_transform_inplace(adata: ad.AnnData, scale_factor: float = 1e4) -> None:
    X = adata.X.tocsr().astype(np.float32) if sparse.issparse(adata.X) else sparse.csr_matrix(np.asarray(adata.X, dtype=np.float32))

    cell_sums = np.asarray(X.sum(axis=1)).ravel()
    cell_sums[cell_sums == 0] = 1.0
    tf = X.multiply(1.0 / cell_sums[:, None])

    n_cells = X.shape[0]
    feature_counts = np.asarray((X > 0).sum(axis=0)).ravel()
    idf = np.log1p(n_cells / (1.0 + feature_counts)).astype(np.float32)

    X_tfidf = tf.multiply(idf)
    X_tfidf = X_tfidf * scale_factor
    X_tfidf.data = np.log1p(X_tfidf.data)
    adata.X = X_tfidf.tocsr()

def normalize_log1p_inplace(adata: ad.AnnData, target_sum: float = 1e4) -> None:
    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)

def add_counts_layer_if_missing(adata: ad.AnnData) -> None:
    adata.layers["counts"] = adata.X.copy()

def maybe_calculate_rna_qc(adata: ad.AnnData, mt_prefix: str = "MT-") -> None:
    if "mt" not in adata.var.columns:
        adata.var["mt"] = adata.var_names.str.startswith(mt_prefix)
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

def calculate_basic_qc(adata: ad.AnnData) -> None:
    sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)

def choose_hvg_inplace(
    adata: ad.AnnData,
    batch_key: str | None = None,
    min_mean: float = 0.0125,
    max_mean: float = 3.0,
    min_disp: float = 0.5,
) -> None:
    kwargs = dict(min_mean=min_mean, max_mean=max_mean, min_disp=min_disp)
    if batch_key is not None and batch_key in adata.obs.columns:
        kwargs["batch_key"] = batch_key
    sc.pp.highly_variable_genes(adata, **kwargs)

def require_obs_column(adata: ad.AnnData, column: str, default_value: str) -> None:
    if column not in adata.obs.columns:
        adata.obs[column] = default_value

def intersect_features(*feature_lists: Sequence[str]) -> List[str]:
    if not feature_lists:
        return []
    common = set(feature_lists[0])
    for x in feature_lists[1:]:
        common &= set(x)
    return sorted(common)

def union_then_intersect(base_a: Sequence[str], base_b: Sequence[str], available_a: Sequence[str], available_b: Sequence[str]) -> List[str]:
    union_set = set(base_a) | set(base_b)
    return sorted(union_set & set(available_a) & set(available_b))

@dataclass
class PartialModalityAssignment:
    modality: Dict[str, str]
    paired_cells: List[str]
    rna_only_cells: List[str]
    atac_only_cells: List[str]

def assign_partial_modality(
    cells: Sequence[str],
    single_frac: float = 0.2,
    rna_keep_prob: float = 0.5,
    seed: int = 1234
) -> PartialModalityAssignment:
    rng = np.random.default_rng(seed)
    cells = np.asarray(list(cells), dtype=object)
    n = len(cells)
    n_single = int(round(n * single_frac))

    if n_single > 0:
        single_cells = rng.choice(cells, size=n_single, replace=False)
    else:
        single_cells = np.array([], dtype=object)

    paired_cells = sorted(set(cells.tolist()) - set(single_cells.tolist()))
    modality = {c: "paired" for c in cells.tolist()}

    if n_single > 0:
        single_is_rna = rng.random(n_single) < rna_keep_prob
        for c, keep_rna in zip(single_cells.tolist(), single_is_rna.tolist()):
            modality[c] = "RNA" if keep_rna else "ATAC"

    rna_only = sorted([c for c, m in modality.items() if m == "RNA"])
    atac_only = sorted([c for c, m in modality.items() if m == "ATAC"])

    return PartialModalityAssignment(
        modality=modality,
        paired_cells=paired_cells,
        rna_only_cells=rna_only,
        atac_only_cells=atac_only,
    )


In [10]:
# 4. Read 10x multiome H5 the R-style way

def read_10x_multiome_h5(input_h5: os.PathLike) -> Tuple[ad.AnnData, ad.AnnData]:
    """
    Read a 10x multiome filtered_feature_bc_matrix.h5 and split into:
    - RNA AnnData
    - ATAC AnnData

    Equivalent in spirit to the R workflow:
        inputdata.10x <- Read10X_h5(input_h5)
        rna_counts  <- inputdata.10x$`Gene Expression`
        atac_counts <- inputdata.10x$Peaks
    """
    adata_all = sc.read_10x_h5(str(input_h5), gex_only=False)

    if "feature_types" not in adata_all.var.columns:
        raise ValueError("The 10x H5 file does not contain var['feature_types']; cannot split Gene Expression and Peaks.")

    feat_types = adata_all.var["feature_types"].astype(str)
    rna_mask = feat_types == "Gene Expression"
    atac_mask = feat_types == "Peaks"

    if rna_mask.sum() == 0:
        raise ValueError("No 'Gene Expression' features found in the 10x H5 file.")
    if atac_mask.sum() == 0:
        raise ValueError("No 'Peaks' features found in the 10x H5 file.")

    rna = adata_all[:, rna_mask].copy()
    atac = adata_all[:, atac_mask].copy()

    # keep names clean
    rna.var_names_make_unique()
    atac.var_names_make_unique()

    # mimic R common cell intersection, though usually already identical
    common_cells = np.intersect1d(rna.obs_names, atac.obs_names)
    rna = rna[common_cells].copy()
    atac = atac[common_cells].copy()

    return rna, atac


In [11]:
# 5. Preprocessing functions

def preprocess_rna_adata(
    rna: ad.AnnData,
    out_path: os.PathLike,
    batch_key: str = "batch",
    mt_prefix: str = "MT-",
) -> ad.AnnData:
    rna = rna.copy()
    require_obs_column(rna, batch_key, "batch0")
    maybe_calculate_rna_qc(rna, mt_prefix=mt_prefix)
    add_counts_layer_if_missing(rna)
    normalize_log1p_inplace(rna)
    choose_hvg_inplace(rna, batch_key=batch_key, min_mean=0.02, max_mean=4.0, min_disp=0.5)
    safe_write_h5ad(rna, out_path)
    return rna

def preprocess_atac_adata(
    atac: ad.AnnData,
    out_path: os.PathLike,
    batch_key: str = "batch",
) -> ad.AnnData:
    atac = atac.copy()
    require_obs_column(atac, batch_key, "batch0")
    calculate_basic_qc(atac)
    add_counts_layer_if_missing(atac)
    tfidf_transform_inplace(atac, scale_factor=1e4)
    choose_hvg_inplace(atac, batch_key=batch_key, min_mean=0.05, max_mean=1.5, min_disp=0.5)
    safe_write_h5ad(atac, out_path)
    return atac

def build_gene_activity(
    atac_qc_path,
    gtf_path,
    out_path,
    batch_key="batch",
):
    import episcanpy as epi
    import numpy as np
    from scipy import sparse

    atac = sc.read_h5ad(str(atac_qc_path))
    if "counts" not in atac.layers:
        raise ValueError("ATAC_counts_qc.h5ad does not contain layers['counts'] needed for gene activity.")

    atac.layers["normalized"] = atac.X.copy()
    atac.X = atac.layers["counts"].copy()

    atac_gas = epi.tl.geneactivity(atac, str(gtf_path), annotation="HAVANA")
    atac_gas = atac_gas[:, ~atac_gas.var_names.duplicated()].copy()

    require_obs_column(atac_gas, batch_key, "batch0")

    # 先把 X 转成可写格式，再存入 layer
    if isinstance(atac_gas.X, np.matrix):
        atac_gas.X = np.asarray(atac_gas.X)

    add_counts_layer_if_missing(atac_gas)
    normalize_log1p_inplace(atac_gas)
    choose_hvg_inplace(atac_gas, batch_key=batch_key)

    # 再统一检查 X 和所有 layers，防止 numpy.matrix 残留
    if isinstance(atac_gas.X, np.matrix):
        atac_gas.X = np.asarray(atac_gas.X)

    for layer_name in list(atac_gas.layers.keys()):
        layer = atac_gas.layers[layer_name]
        if isinstance(layer, np.matrix):
            atac_gas.layers[layer_name] = np.asarray(layer)

    safe_write_h5ad(atac_gas, out_path)
    return atac_gas

def build_feature_aligned(
    rna_qc_path: os.PathLike,
    atac_gas_path: os.PathLike,
    out_path: os.PathLike,
) -> ad.AnnData:
    rna = sc.read_h5ad(str(rna_qc_path))
    atac_gas = sc.read_h5ad(str(atac_gas_path))

    rna_hvg = rna.var_names[rna.var["highly_variable"].fillna(False)].tolist()
    atac_hvg = atac_gas.var_names[atac_gas.var["highly_variable"].fillna(False)].tolist()

    aligned_genes = union_then_intersect(
        rna_hvg,
        atac_hvg,
        rna.var_names.tolist(),
        atac_gas.var_names.tolist(),
    )
    if len(aligned_genes) == 0:
        raise ValueError("No common aligned genes between RNA HVGs and ATAC-gene-activity HVGs.")

    rna_sub = rna[:, aligned_genes].copy()
    atac_sub = atac_gas[:, aligned_genes].copy()

    rna_sub.obs["modality"] = "RNA"
    atac_sub.obs["modality"] = "ATAC_gene_activity"

    aligned = ad.concat(
        [rna_sub, atac_sub],
        join="inner",
        label="modality_concat",
        keys=["RNA", "ATAC_gene_activity"],
        index_unique="__"
    )
    aligned.uns["rna_hvg"] = rna_hvg
    aligned.uns["atac_hvg"] = atac_hvg
    aligned.uns["aligned_genes"] = aligned_genes

    safe_write_h5ad(aligned, out_path)
    return aligned


In [12]:
# 6. Split generation, matching the R logic

def subset_and_copy(
    adata: ad.AnnData,
    cells: Sequence[str],
    features: Sequence[str] | None = None
) -> ad.AnnData:
    out = adata[list(cells)].copy()
    if features is not None:
        out = out[:, list(features)].copy()
    return out


def save_split_h5ads(
    outdir: os.PathLike,
    train_rna_ref: ad.AnnData,
    train_atac_full: ad.AnnData,
    train_atac_activity: ad.AnnData,
    val_query_atac: ad.AnnData,
    val_true_rna: ad.AnnData,
    val_atac_activity: ad.AnnData,
) -> None:
    outdir = Path(outdir)
    safe_write_h5ad(train_rna_ref, outdir / "train_rna_ref.h5ad")
    safe_write_h5ad(train_atac_full, outdir / "train_atac_full.h5ad")
    safe_write_h5ad(train_atac_activity, outdir / "train_atac_activity.h5ad")
    safe_write_h5ad(val_query_atac, outdir / "val_query_atac.h5ad")
    safe_write_h5ad(val_true_rna, outdir / "val_true_rna.h5ad")
    safe_write_h5ad(val_atac_activity, outdir / "val_atac_activity.h5ad")


def generate_splits(
    rna_qc_path: os.PathLike,
    atac_qc_path: os.PathLike,
    atac_gas_path: os.PathLike,
    out_root: os.PathLike,
    seed: int = 1234,
    val_frac: float = 0.2,
    single_fracs: Sequence[float] = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    rna_keep_prob: float = 0.5,
    min_train_rna_cells: int = 50,
    min_val_query_cells: int = 20,
    min_common_features: int = 50,
) -> pd.DataFrame:
    rna = sc.read_h5ad(str(rna_qc_path))
    atac_peak = sc.read_h5ad(str(atac_qc_path))
    atac_gas = sc.read_h5ad(str(atac_gas_path))

    # 细胞交集：三者都要有
    common_cells = intersect_features(
        rna.obs_names.tolist(),
        atac_peak.obs_names.tolist(),
        atac_gas.obs_names.tolist()
    )
    if len(common_cells) == 0:
        raise ValueError("No common cells shared by RNA_counts_qc, ATAC_counts_qc, and ATAC_gas.")

    rna = rna[common_cells].copy()
    atac_peak = atac_peak[common_cells].copy()
    atac_gas = atac_gas[common_cells].copy()

    # RNA 与 gene activity 的共同基因特征
    common_features = intersect_features(
        rna.var_names.tolist(),
        atac_gas.var_names.tolist()
    )
    if len(common_features) < min_common_features:
        raise ValueError(
            f"Too few common gene-level features between RNA and ATAC_gas: {len(common_features)}"
        )

    rna = rna[:, common_features].copy()
    atac_gas = atac_gas[:, common_features].copy()

    all_cells = np.array(common_cells, dtype=object)
    train_cells, val_cells = train_test_split(
        all_cells,
        test_size=val_frac,
        random_state=seed,
        shuffle=True
    )
    train_cells = np.array(sorted(train_cells.tolist()), dtype=object)
    val_cells = np.array(sorted(val_cells.tolist()), dtype=object)

    ensure_dir(out_root)
    summaries = []

    for sf in single_fracs:
        ratio_label = f"single_{int(round(sf * 100)):03d}"
        outdir = Path(out_root) / ratio_label
        ensure_dir(outdir)

        train_assign = assign_partial_modality(
            train_cells,
            single_frac=sf,
            rna_keep_prob=rna_keep_prob,
            seed=seed + int(round(sf * 1000)) + 11
        )
        val_assign = assign_partial_modality(
            val_cells,
            single_frac=sf,
            rna_keep_prob=rna_keep_prob,
            seed=seed + int(round(sf * 1000)) + 97
        )

        # 和 R 逻辑保持一致
        train_rna_ref_cells = sorted(train_assign.paired_cells + train_assign.rna_only_cells)
        train_atac_ref_cells = sorted(train_assign.paired_cells + train_assign.atac_only_cells)
        val_query_atac_cells = sorted(val_assign.paired_cells + val_assign.atac_only_cells)

        if (
            len(train_rna_ref_cells) < min_train_rna_cells
            or len(train_atac_ref_cells) < min_train_rna_cells
            or len(val_query_atac_cells) < min_val_query_cells
        ):
            meta = {
                "ratio_label": ratio_label,
                "single_frac": sf,
                "status": "skipped",
                "reason": "too_few_cells",
                "train_rna_ref": len(train_rna_ref_cells),
                "train_atac_ref": len(train_atac_ref_cells),
                "val_query_atac": len(val_query_atac_cells),
            }
            with open(outdir / "split_info.json", "w", encoding="utf-8") as f:
                json.dump(meta, f, indent=2, ensure_ascii=False)
            summaries.append(meta)
            continue

        # -------- 保存各类 split 文件 --------
        train_rna_ref = subset_and_copy(rna, train_rna_ref_cells)

        # train ATAC full: 保留整套训练 ATAC peak-level 数据（沿用你原来的命名）
        train_atac_full = subset_and_copy(atac_peak, train_cells)

        # 新增：train ATAC activity，只取 train ATAC reference cells
        train_atac_activity = subset_and_copy(atac_gas, train_atac_ref_cells)

        # val query peak-level ATAC
        val_query_atac = subset_and_copy(atac_peak, val_query_atac_cells)

        # val true RNA
        val_true_rna = subset_and_copy(rna, val_query_atac_cells)

        # val ATAC activity
        val_atac_activity = subset_and_copy(atac_gas, val_query_atac_cells)

        save_split_h5ads(
            outdir,
            train_rna_ref=train_rna_ref,
            train_atac_full=train_atac_full,
            train_atac_activity=train_atac_activity,
            val_query_atac=val_query_atac,
            val_true_rna=val_true_rna,
            val_atac_activity=val_atac_activity,
        )

        meta = {
            "ratio_label": ratio_label,
            "single_frac": sf,
            "status": "ok",
            "train_cells": train_cells.tolist(),
            "val_cells": val_cells.tolist(),
            "train_paired_cells": train_assign.paired_cells,
            "train_rna_only_cells": train_assign.rna_only_cells,
            "train_atac_only_cells": train_assign.atac_only_cells,
            "val_paired_cells": val_assign.paired_cells,
            "val_rna_only_cells": val_assign.rna_only_cells,
            "val_atac_only_cells": val_assign.atac_only_cells,
            "train_rna_ref_cells": train_rna_ref_cells,
            "train_atac_ref_cells": train_atac_ref_cells,
            "val_query_atac_cells": val_query_atac_cells,
            "common_gene_features": common_features,
            "n_train_rna_ref_cells": len(train_rna_ref_cells),
            "n_train_atac_ref_cells": len(train_atac_ref_cells),
            "n_val_query_atac_cells": len(val_query_atac_cells),
            "n_common_features": len(common_features),
        }
        with open(outdir / "split_info.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, indent=2, ensure_ascii=False)

        summaries.append({
            "ratio_label": ratio_label,
            "single_frac": sf,
            "status": "ok",
            "train_paired": len(train_assign.paired_cells),
            "train_rna_only": len(train_assign.rna_only_cells),
            "train_atac_only": len(train_assign.atac_only_cells),
            "val_paired": len(val_assign.paired_cells),
            "val_rna_only": len(val_assign.rna_only_cells),
            "val_atac_only": len(val_assign.atac_only_cells),
            "train_rna_ref": len(train_rna_ref_cells),
            "train_atac_ref": len(train_atac_ref_cells),
            "val_query_atac": len(val_query_atac_cells),
            "n_common_features": len(common_features),
        })

    summary_df = pd.DataFrame(summaries)
    summary_df.to_csv(Path(out_root) / "summary_all_ratios.csv", index=False)
    return summary_df

In [13]:
# 7. Build or locate the four base files

set_seed(SEED)
ensure_dir(OUTPUT_DIR)

output_dir = Path(OUTPUT_DIR)
rna_qc_path = output_dir / "RNA_counts_qc.h5ad"
atac_qc_path = output_dir / "ATAC_counts_qc.h5ad"
atac_gas_path = output_dir / "ATAC_gas.h5ad"
feature_aligned_path = output_dir / "feature_aligned.h5ad"

if INPUT_MODE == "from_10x_h5":
    print("Reading 10x multiome H5 ...")
    rna_raw, atac_raw = read_10x_multiome_h5(INPUT_H5)

    print("RNA raw shape:", rna_raw.shape)
    print("ATAC raw shape:", atac_raw.shape)
    print("FRAGMENTS_FILE (kept for compatibility):", FRAGMENTS_FILE)

    print("[1/4] Preprocessing RNA ...")
    preprocess_rna_adata(rna_raw, rna_qc_path, batch_key=BATCH_KEY, mt_prefix=MT_PREFIX)

    print("[2/4] Preprocessing ATAC ...")
    preprocess_atac_adata(atac_raw, atac_qc_path, batch_key=BATCH_KEY)

    print("[3/4] Building ATAC gene activity ...")
    build_gene_activity(atac_qc_path, GTF_PATH, atac_gas_path, batch_key=BATCH_KEY)

    print("[4/4] Building feature-aligned file ...")
    build_feature_aligned(rna_qc_path, atac_gas_path, feature_aligned_path)

elif INPUT_MODE == "from_existing_h5ad":
    base_dir = Path(EXISTING_BASE_DIR)

    rna_qc_path = base_dir / "RNA_counts_qc.h5ad"
    atac_qc_path = base_dir / "ATAC_counts_qc.h5ad"
    atac_gas_path = base_dir / "ATAC_gas.h5ad"
    feature_aligned_path = base_dir / "feature_aligned.h5ad"

    missing = [p.name for p in [rna_qc_path, atac_qc_path, atac_gas_path, feature_aligned_path] if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing base files in EXISTING_BASE_DIR: " + ", ".join(missing))

    print("Using existing h5ad base files:")
    print(rna_qc_path)
    print(atac_qc_path)
    print(atac_gas_path)
    print(feature_aligned_path)

else:
    raise ValueError("INPUT_MODE must be either 'from_10x_h5' or 'from_existing_h5ad'")


Reading 10x multiome H5 ...


/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


RNA raw shape: (11909, 36601)
ATAC raw shape: (11909, 108377)
FRAGMENTS_FILE (kept for compatibility): raw_input/pbmc_granulocyte_sorted_10k_atac_fragments.tsv.gz
[1/4] Preprocessing RNA ...
[2/4] Preprocessing ATAC ...
[3/4] Building ATAC gene activity ...


[4/4] Building feature-aligned file ...


In [14]:
# 8. Generate train / validation split files

summary_df = generate_splits(
    rna_qc_path=rna_qc_path,
    atac_qc_path=atac_qc_path,
    atac_gas_path=atac_gas_path,
    out_root=Path(OUTPUT_DIR) / SPLIT_DIR_NAME,
    seed=SEED,
    val_frac=VAL_FRAC,
    single_fracs=SINGLE_FRACS,
    rna_keep_prob=RNA_KEEP_PROB,
    min_train_rna_cells=MIN_TRAIN_RNA_CELLS,
    min_val_query_cells=MIN_VAL_QUERY_CELLS,
    min_common_features=MIN_COMMON_FEATURES,
)

summary_df


,ratio_label,single_frac,status,train_paired,train_rna_only,train_atac_only,val_paired,val_rna_only,val_atac_only,train_rna_ref,train_atac_ref,val_query_atac,n_common_features
0,single_000,0.0,ok,9527,0,0,2382,0,0,9527,9527,2382,19115
1,single_020,0.2,ok,7622,944,961,1906,250,226,8566,8583,2132,19115
2,single_040,0.4,ok,5716,1952,1859,1429,446,507,7668,7575,1936,19115
3,single_060,0.6,ok,3811,2826,2890,953,685,744,6637,6701,1697,19115
4,single_080,0.8,ok,1905,3816,3806,476,966,940,5721,5711,1416,19115
5,single_100,1.0,ok,0,4654,4873,0,1156,1226,4654,4873,1226,19115


In [15]:
# 9. Quick file check

for p in sorted(Path(OUTPUT_DIR).rglob("*")):
    print(p)


output_pbmc_multiome/ATAC_counts_qc.h5ad
output_pbmc_multiome/ATAC_gas.h5ad
output_pbmc_multiome/RNA_counts_qc.h5ad
output_pbmc_multiome/feature_aligned.h5ad
output_pbmc_multiome/results_ratio_loop
output_pbmc_multiome/results_ratio_loop/single_000
output_pbmc_multiome/results_ratio_loop/single_000/split_info.json
output_pbmc_multiome/results_ratio_loop/single_000/train_atac_activity.h5ad
output_pbmc_multiome/results_ratio_loop/single_000/train_atac_full.h5ad
output_pbmc_multiome/results_ratio_loop/single_000/train_rna_ref.h5ad
output_pbmc_multiome/results_ratio_loop/single_000/val_atac_activity.h5ad
output_pbmc_multiome/results_ratio_loop/single_000/val_query_atac.h5ad
output_pbmc_multiome/results_ratio_loop/single_000/val_true_rna.h5ad
output_pbmc_multiome/results_ratio_loop/single_020
output_pbmc_multiome/results_ratio_loop/single_020/split_info.json
output_pbmc_multiome/results_ratio_loop/single_020/train_atac_activity.h5ad
output_pbmc_multiome/results_ratio_loop/single_020/train_a

## Outputs

### Four base files
- `RNA_counts_qc.h5ad`
- `ATAC_counts_qc.h5ad`
- `ATAC_gas.h5ad`
- `feature_aligned.h5ad`

### Split files per ratio
Inside:
- `results_ratio_loop/single_000/`
- `results_ratio_loop/single_020/`
- ...
- `results_ratio_loop/single_100/`

Each contains:
- `train_rna_ref.h5ad`
- `train_atac_full.h5ad`
- `val_query_atac.h5ad`
- `val_true_rna.h5ad`
- `val_atac_activity.h5ad`
- `split_info.json`

### Important
This notebook now supports both:
1. **R-style 10x multiome H5 input**
2. **existing Python-generated H5AD outputs**


In [ ]:
import scanpy as sc

adata_post = sc.read_h5ad("/data5/zhangye/scMRDR/scMRDR_zy/PBMC/output_pbmc_multiome/results_scMRDR_loop/single_000/training_adata_post.h5ad")

print("layers:", list(adata_post.layers.keys()))
print("obsm:", list(adata_post.obsm.keys()))
print("uns:", list(adata_post.uns.keys()))
print("shape:", adata_post.shape)

layers: ['count']
obsm: ['latent_shared', 'latent_specific']
uns: []
shape: (21436, 19115)


/home/zhangye/anaconda3/envs/scmrdr/lib/python3.9/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
